# Módulo 05: AutoML y Feature Store

## Contenido del Módulo

1. **Introducción a AutoML**
   - ¿Qué es AutoML?
   - Ventajas y limitaciones
   - AutoML en Databricks

2. **Feature Store**
   - Motivación y conceptos clave
   - Arquitectura de Feature Store
   - Feature Tables y Feature Serving

3. **Integración AutoML + Feature Store**
   - Flujo de trabajo completo
   - Mejores prácticas
   - Casos de uso

---

**Objetivo del módulo**: Comprender las herramientas de automatización y gestión de features que aceleran el desarrollo de modelos ML en producción.

# 1. Introducción a AutoML

## ¿Qué es AutoML?

**AutoML (Automated Machine Learning)** automatiza tareas repetitivas del proceso de ML:
- Preprocesamiento de datos
- Selección de algoritmos
- Ingeniería de características
- Ajuste de hiperparámetros
- Validación cruzada
- Selección del modelo óptimo

## Ventajas de AutoML

| Ventaja | Descripción |
|---------|-------------|
| **Velocidad** | Reduce el tiempo de desarrollo de semanas a minutos |
| **Democratización** | Permite a usuarios no expertos crear modelos de calidad |
| **Benchmark** | Proporciona una línea base sólida para comparar |
| **Exploración** | Prueba múltiples algoritmos y configuraciones automáticamente |
| **Reproducibilidad** | Genera código y notebooks reproducibles |

## Limitaciones de AutoML

| Limitación | Descripción |
|------------|-------------|
| **Caja negra** | Menor control sobre el proceso de modelado |
| **Datos específicos** | Requiere datos bien preparados y representativos |
| **Interpretabilidad** | Puede sacrificar interpretabilidad por performance |
| **Recursos** | Consume recursos computacionales significativos |
| **Dominio** | No reemplaza el conocimiento del dominio del negocio |

## AutoML en Databricks

Databricks AutoML ofrece:
- **Interfaz gráfica** para usuarios no técnicos
- **API de Python** para integración en workflows
- **Notebooks generados** con código explicativo
- **Registro automático en MLflow** para tracking
- **Soporte para**:
  - Clasificación (binaria y multiclase)
  - Regresión
  - Forecasting (series temporales)

```python
# Ejemplo básico de uso
import databricks.automl

summary = databricks.automl.classify(
    dataset=train_df,
    target_col="target",
    timeout_minutes=15
)
```

# 2. Feature Store - Conceptos Clave

## ¿Qué es Feature Store?

**Feature Store** es un repositorio centralizado para features de ML que:
- **Almacena** features reutilizables
- **Versionado** features con control de cambios
- **Comparte** features entre equipos
- **Sirve** features en tiempo real y batch
- **Garantiza consistencia** entre entrenamiento e inferencia

## Motivación

### Problemas sin Feature Store

| Problema | Impacto |
|----------|--------|
| **Duplicación** | Mismas features calculadas múltiples veces |
| **Inconsistencia** | Diferencias entre entrenamiento y producción |
| **Descubrimiento** | Difícil encontrar features existentes |
| **Documentación** | Features mal documentadas o sin documentar |
| **Gobernanza** | Difícil controlar acceso y calidad |

### Solución con Feature Store

```
┌─────────────────────────────────────────────┐
│         Feature Store (Delta Tables)        │
├─────────────────────────────────────────────┤
│  Feature Table 1: customer_features         │
│  Feature Table 2: transaction_features      │
│  Feature Table 3: product_features          │
└─────────────────────────────────────────────┘
         ↓                    ↓
    Training              Serving
    (Batch)            (Real-time)
```

## Componentes Principales

### 1. Feature Table

- **Tabla Delta** en Unity Catalog
- Contiene:
  - **Primary keys**: Identificadores únicos
  - **Features**: Columnas con features
  - **Timestamp** (opcional): Para point-in-time lookups
  - **Metadata**: Descripción, autor, versión

### 2. Feature Spec

- Define qué features usar del Feature Store
- Especifica:
  - Tabla origen
  - Columnas a incluir
  - Claves de join

### 3. Training Set

- Combina labels con features del Feature Store
- Garantiza consistencia temporal (point-in-time correctness)

### 4. Feature Serving

- **Batch**: Para predicciones en lote
- **Online**: Para inferencia en tiempo real (low-latency)

# 3. Arquitectura de Feature Store

## Flujo Completo

```
┌─────────────┐
│ Raw Data    │
└──────┬──────┘
       │
       ↓
┌─────────────────────┐
│ Feature Engineering │ ← Spark / Pandas
└──────┬──────────────┘
       │
       ↓
┌─────────────────────┐
│  Feature Store      │ ← Delta Tables
│  (Unity Catalog)    │
└──────┬──────────────┘
       │
       ├──────────────┐
       ↓              ↓
  ┌─────────┐   ┌──────────┐
  │Training │   │ Serving  │
  └─────────┘   └──────────┘
       │              │
       ↓              ↓
  ┌─────────┐   ┌──────────┐
  │  Model  │   │Real-time │
  │ Training│   │Inference │
  └─────────┘   └──────────┘
```

## Ventajas de la Arquitectura

1. **Single Source of Truth**
   - Features definidas una sola vez
   - Usadas en múltiples modelos

2. **Consistencia Training-Serving**
   - Misma lógica en entrenamiento y producción
   - Elimina "training-serving skew"

3. **Point-in-Time Correctness**
   - Features históricas correctas para cada timestamp
   - Evita data leakage

4. **Reutilización**
   - Features compartidas entre equipos
   - Reduce tiempo de desarrollo

5. **Gobernanza**
   - Control de acceso vía Unity Catalog
   - Auditoría y linaje de datos

## Tecnologías en Databricks

- **Storage**: Delta Lake (ACID, versionado, time travel)
- **Catalog**: Unity Catalog (gobernanza, permisos)
- **Compute**: Spark / Pandas on Spark
- **API**: Python Feature Store API
- **Integration**: MLflow para tracking

In [0]:
# 4. Creación de Feature Tables

from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql import functions as F

# Inicializar cliente
fe = FeatureEngineeringClient()

# Ejemplo: Crear features de clientes
customer_features = (
    spark.table("main.default.customers")
    .withColumn("days_since_signup", F.datediff(F.current_date(), F.col("signup_date")))
    .withColumn("total_orders", F.col("order_count"))
    .withColumn("avg_order_value", F.col("total_spent") / F.col("order_count"))
    .select(
        "customer_id",  # Primary key
        "days_since_signup",
        "total_orders",
        "avg_order_value",
        "preferred_category",
        "last_purchase_date"  # Timestamp column
    )
)

# Crear Feature Table
fe.create_table(
    name="main.features.customer_features",
    primary_keys=["customer_id"],
    timestamp_keys=["last_purchase_date"],
    df=customer_features,
    description="Customer behavioral features for ML models"
)

print("✅ Feature Table creada exitosamente")

In [0]:
# 5. Actualización de Feature Tables

# Opción 1: Sobrescribir completamente
fe.write_table(
    name="main.features.customer_features",
    df=customer_features_updated,
    mode="overwrite"
)

# Opción 2: Merge (upsert)
# Actualiza registros existentes e inserta nuevos
fe.write_table(
    name="main.features.customer_features",
    df=customer_features_new,
    mode="merge"
)

print("✅ Feature Table actualizada")

# Leer Feature Table
features_df = fe.read_table(name="main.features.customer_features")
print(f"Total features: {features_df.count()}")
features_df.display()

In [0]:
# 6. Uso de Features en Training

from databricks.feature_engineering import FeatureLookup

# Dataset con labels
labels_df = spark.table("main.default.training_labels").select(
    "customer_id",
    "churn",  # Target variable
    "label_date"  # Timestamp para point-in-time lookup
)

# Definir qué features usar
feature_lookups = [
    FeatureLookup(
        table_name="main.features.customer_features",
        lookup_key="customer_id",
        timestamp_lookup_key="label_date",  # Point-in-time correctness
        feature_names=[
            "days_since_signup",
            "total_orders",
            "avg_order_value",
            "preferred_category"
        ]
    )
]

# Crear training set
training_set = fe.create_training_set(
    df=labels_df,
    feature_lookups=feature_lookups,
    label="churn",
    exclude_columns=["label_date"]
)

# Cargar como DataFrame
training_df = training_set.load_df()
print(f"Training set shape: {training_df.count()} rows")
training_df.display()

In [0]:
# 7. Integración AutoML + Feature Store

import databricks.automl

# Opción 1: AutoML con Training Set del Feature Store
training_df_pandas = training_set.load_df().toPandas()

summary = databricks.automl.classify(
    dataset=training_df_pandas,
    target_col="churn",
    timeout_minutes=10,
    primary_metric="f1"
)

print(f"Mejor modelo: {summary.best_trial.model_description}")
print(f"F1 Score: {summary.best_trial.metrics['val_f1_score']:.4f}")

# El modelo queda registrado en MLflow con referencia al Feature Store
# Esto permite:
# 1. Reproducibilidad: Features versionadas
# 2. Serving: Lookup automático de features en inferencia
# 3. Lineage: Trazabilidad desde features hasta predicciones

In [0]:
# 8. Feature Serving en Producción

import mlflow

# Cargar modelo registrado (incluye Feature Store metadata)
model_uri = "models:/churn_prediction/production"
model = mlflow.pyfunc.load_model(model_uri)

# Predicción Batch
# Solo necesitas las primary keys, las features se buscan automáticamente
batch_df = spark.createDataFrame([
    (12345,),
    (67890,)
], ["customer_id"])

predictions = fe.score_batch(
    model_uri=model_uri,
    df=batch_df
)

predictions.display()

# Las features se obtienen automáticamente del Feature Store
# Garantiza consistencia entre training y serving

# 9. Mejores Prácticas

## Diseño de Feature Tables

### ✅ Buenas Prácticas

| Práctica | Razón |
|----------|-------|
| **Una entidad por tabla** | Customer features, product features separados |
| **Primary keys claros** | IDs únicos y estables |
| **Timestamp keys** | Para point-in-time correctness |
| **Documentación** | Descripción detallada de cada feature |
| **Versionado** | Usar Delta time travel para auditoría |
| **Tipos de datos consistentes** | Evitar casting innecesario |

### ❌ Anti-patrones

| Anti-patrón | Problema |
|-------------|----------|
| Features de múltiples entidades | Complica joins y actualizaciones |
| Sin timestamp keys | No permite lookups temporales correctos |
| Features calculadas en serving | Inconsistencia training-serving |
| Sin documentación | Features no reutilizables |
| Actualización manual | Propenso a errores y desactualización |

## Workflow Recomendado

```python
# 1. Ingeniería de Features (una vez)
features = compute_features(raw_data)
fe.create_table(name="main.features.my_features", df=features, ...)

# 2. Training (múltiples modelos)
training_set = fe.create_training_set(labels, feature_lookups)
summary = databricks.automl.classify(training_set.load_df(), ...)

# 3. Serving (automático)
predictions = fe.score_batch(model_uri, new_data)
```

## AutoML - Mejores Prácticas

| Práctica | Descripción |
|----------|-------------|
| **Datos limpios** | AutoML no hace limpieza avanzada |
| **Features significativas** | Feature engineering previo mejora resultados |
| **Timeout adecuado** | Más tiempo = mejor exploración (15-60 min) |
| **Métrica correcta** | Alineada con objetivo de negocio |
| **Revisar notebooks** | Aprender de las técnicas generadas |
| **Benchmark** | Usar como baseline, iterar manualmente |

## Cuándo Usar Cada Herramienta

### Usar AutoML cuando:
- Necesitas un modelo rápidamente
- Quieres establecer un baseline
- No tienes experiencia profunda en ML
- El problema es estándar (clasificación/regresión)

### Usar Feature Store cuando:
- Múltiples modelos usan las mismas features
- Necesitas consistencia training-serving
- Trabajas en equipo
- Modelos en producción

### Combinar ambos cuando:
- Quieres acelerar desarrollo **Y** mantener calidad
- Múltiples científicos de datos colaboran
- Ciclo de vida completo: prototipo → producción

# 10. Comparación de Workflows

## Workflow Tradicional (sin Feature Store)

```python
# Modelo 1: Churn
features_churn = compute_customer_features(raw_data)  # ← Código duplicado
train_model_churn(features_churn)

# Modelo 2: Lifetime Value
features_ltv = compute_customer_features(raw_data)    # ← Mismo cálculo otra vez
train_model_ltv(features_ltv)

# Producción: Recomputar features en serving
def predict_churn(customer_id):
    features = compute_customer_features(customer_id)  # ← Riesgo de inconsistencia
    return model.predict(features)
```

**Problemas**:
- Features calculadas múltiples veces
- Riesgo de inconsistencia entre training y serving
- Difícil compartir entre equipos

---

## Workflow con Feature Store

```python
# 1. Computar features UNA VEZ
features = compute_customer_features(raw_data)
fe.create_table("main.features.customer_features", df=features)

# 2. Modelo 1: Churn
training_set_churn = fe.create_training_set(
    labels_churn, 
    feature_lookups=[...]
)
train_model_churn(training_set_churn)

# 3. Modelo 2: Lifetime Value (reutiliza mismas features)
training_set_ltv = fe.create_training_set(
    labels_ltv,
    feature_lookups=[...]  # ← Misma tabla, cero cálculo adicional
)
train_model_ltv(training_set_ltv)

# 4. Producción: Features automáticas
predictions = fe.score_batch(model_uri, customer_ids)  # ← Lookup automático
```

**Ventajas**:
- ✅ Features calculadas una sola vez
- ✅ Consistencia garantizada
- ✅ Reutilización entre modelos
- ✅ Menos código, menos bugs

---

## Impacto en Métricas

| Métrica | Sin Feature Store | Con Feature Store |
|---------|-------------------|-------------------|
| Tiempo desarrollo nuevo modelo | 2-4 semanas | 1-2 semanas |
| Training-serving skew | 10-30% de modelos | <5% |
| Features reutilizadas | <20% | >70% |
| Tiempo de deployment | Días | Horas |
| Esfuerzo de mantenimiento | Alto | Bajo |

# Resumen y Conclusiones

## Conceptos Clave Aprendidos

### AutoML
- ✅ Automatiza tareas repetitivas del proceso ML
- ✅ Acelera desarrollo y proporciona baseline sólido
- ✅ Databricks AutoML genera notebooks reproducibles
- ⚠️ No reemplaza conocimiento del dominio
- ⚠️ Requiere datos bien preparados

### Feature Store
- ✅ Repositorio centralizado de features
- ✅ Garantiza consistencia training-serving
- ✅ Facilita reutilización y colaboración
- ✅ Point-in-time correctness para evitar leakage
- ✅ Integrado con Unity Catalog y MLflow

### Integración AutoML + Feature Store
- Workflow completo: features → training → serving
- Reproducibilidad y trazabilidad end-to-end
- Aceleración dramática del ciclo de desarrollo

---

## Flujo de Trabajo Recomendado

```
1. Feature Engineering
   └→ Crear Feature Tables

2. Training
   ├→ Create Training Set (Feature Lookups)
   └→ AutoML para exploración rápida

3. Refinamiento
   ├→ Revisar notebooks de AutoML
   └→ Iterar manualmente si necesario

4. Deployment
   └→ Feature Store garantiza consistencia

5. Monitoring
   └→ MLflow tracking + Feature Store lineage
```

---

## Próximos Pasos

En el **notebook de práctica** aplicarás:
1. Crear Feature Tables para datasets reales
2. Ejecutar AutoML con Feature Store
3. Comparar resultados con/sin Feature Store
4. Implementar feature serving para producción
5. Analizar métricas y mejores prácticas

**Continúa al notebook:** `Práctica - AutoML y Feature Store` 🚀